# Lab Instructions

Create 3 visualizations from a spatial and time-series dataset of your choice.  Describe your dataset including where it came from and the features it contains.  Each visualization should be accompanied by at least 1 - 2 sentences explaining how the features do (or do not!) change over time and througout space.

# 3.5 Visualizing Relationships in Space and Time Lab
## Dataset Description
**Dataset Source:** rideshare_kaggle.csv (Boston Rideshare Logs).

**Features Included:**
* **Spatial Features:** source (pickup location) and destination (dropoff location).
* **Time-Series/Temporal Features:** hour (hour of day 0–23) and date.
* **Numerical Metrics:** price (fare amount in USD) and distance (trip path distance in miles).

## Python Implementation 

In [ ]:
import pandas as pd
import plotly.express as px

# Load the dataset from csv
df = pd.read_csv('rideshare_kaggle.csv')

# Preprocess time columns for temporal tracking
df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = df['datetime'].dt.date

# Helper grouping to categorize hours into broader time blocks for clarity
def get_time_of_day(hour):
    if 6 <= hour <= 11: return 'Morning'
    elif 12 <= hour <= 16: return 'Afternoon'
    elif 17 <= hour <= 21: return 'Evening'
    else: return 'Night'
df['time_of_day'] = df['hour'].apply(get_time_of_day)


# Interactive Heatmap (Pickup Location vs. Hour of Day)
# Aggregate ride counts by spatial pickup location and temporal hour
df_heatmap = df.groupby(['source', 'hour']).size().reset_index(name='ride_count')

fig1 = px.density_heatmap(
    df_heatmap, 
    x="hour", 
    y="source", 
    z="ride_count",
    title="Spatial-Temporal Demand: Ride Counts by Pickup Location and Hour of Day",
    color_continuous_scale="Viridis",
    labels={'hour': 'Hour of Day', 'source': 'Pickup Location (Spatial)', 'ride_count': 'Number of Rides'}
)
fig1.update_layout(width=900, height=500)
fig1.show()


# Interactive Line Chart (Price Trajectories Over Dates)
# Select a representative set of destinations to map trajectories clearly
top_destinations = ['Back Bay', 'Boston University', 'Financial District', 'Haymarket Square', 'Theatre District']
df_filtered_dest = df[df['destination'].isin(top_destinations)]
df_lines = df_filtered_dest.groupby(['date', 'destination'])['price'].mean().reset_index()

fig2 = px.line(
    df_lines, 
    x="date", 
    y="price", 
    color="destination",
    markers=True,
    title="Temporal Price Fluctuations Across Different Spatial Destinations",
    labels={'date': 'Date', 'price': 'Average Price ($)', 'destination': 'Destination'}
)
fig2.update_layout(width=900, height=500)
fig2.show()


# Interactive Grouped Bar Chart (Travel Distances)
# Aggregate distances by pickup location and time of day blocks
df_bars = df.groupby(['source', 'time_of_day'])['distance'].mean().reset_index()

fig3 = px.bar(
    df_bars, 
    x="source", 
    y="distance", 
    color="time_of_day",
    barmode="group",
    title="Spatial Travel Patterns: Average Ride Distance by Source Location and Time of Day",
    labels={'source': 'Pickup Location (Spatial)', 'distance': 'Average Distance (Miles)', 'time_of_day': 'Time of Day'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig3.update_layout(width=900, height=500, xaxis_tickangle=-45)
fig3.show()

## Interpretations and Visualizations
![](p1.png)
### Visualization 1 Interpretation (Density Heatmap)
This interactive heatmap highlights that passenger demand varies drastically across time, featuring a distinct, uniform dip in rides across all spatial neighborhoods between the early hours of 5 AM and 10 AM, followed by a major surge in demand between 10 AM and 3 PM. This also portrays a massive drop in ride counts from 8PM to midnight. From a spatial standpoint, areas like the North End, Haymarket Square, and Financial District experience the absolute highest concentration of ride volume (bright yellow zones) during these midday peak hours compared to residential boundaries like Beacon Hill.

![](p2.png)
### Visualization 2 Interpretation (Line Chart)
The multi-line visualization illustrates that average fare pricing moves fluidly across time, showing sudden dips around early December that affect all tracked neighborhoods concurrently. However, strict spatial inequality remains constant over the whole period, as trips to Boston University stay anchored at a higher baseline pricing tier than those bound for Haymarket Square.

![](p3.png)
### Visualization 3 Interpretation (Grouped Bar Chart)
This bar chart reveals that spatial variables strictly dictate the travel characteristics of a neighborhood, with regions like Boston University generating significantly longer trip vectors on average than centralized nodes like Haymarket Square. Looking across time categories (Morning, Afternoon, Evening, Night), the trip distances remain mostly flat within each specific neighborhood, revealing that temporal shifts have minimal impact on transit distance bounds.
